# Lesson 33: Survival Analysis

*ML Academy — companion notebook*

Time-to-event data with censoring: Kaplan–Meier curves, median survival, the log-rank test, the Cox proportional hazards model and the concordance index.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-33.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 2. Censored data

In [1]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(4)
n = 600
plan = rng.integers(0, 2, n)                                  # 0 monthly, 1 annual
support = rng.poisson(2, n)                                   # support tickets in the first month
rate = np.exp(-2.0 - 0.8 * plan + 0.25 * support)             # monthly hazard of cancelling
churn_time = rng.exponential(1 / rate)
study_end = rng.uniform(6, 24, n)                             # customers joined at different times
df = pd.DataFrame({"plan": plan, "support": support,
                   "months": np.minimum(churn_time, study_end).round(2),
                   "churned": (churn_time <= study_end).astype(int)})
print(df.head(4).to_string(index=False))
print("customers who churned during the study:", df.churned.sum(), "of", n)
print("naive average of 'months' (ignoring censoring):", round(df.months.mean(), 2), "  true mean time to churn:", round(churn_time.mean(), 2))

 plan  support  months  churned
    1        3    3.71        1
    1        3    7.30        1
    1        4    2.44        1
    1        0    5.78        1
customers who churned during the study: 494 of 600
naive average of 'months' (ignoring censoring): 6.02   true mean time to churn: 7.86


## 3. The Kaplan–Meier estimator

In [2]:
def kaplan_meier(t, e):
    order = np.argsort(t); t, e = t[order], e[order]
    times, surv, s, at_risk = [0.0], [1.0], 1.0, len(t)
    for u in np.unique(t):
        d = e[t == u].sum()                                     # churned at exactly this time
        if d: s *= 1 - d / at_risk; times.append(u); surv.append(s)
        at_risk -= (t == u).sum()                               # churned or censored leave the risk set
    return np.array(times), np.array(surv)
def median_survival(times, surv):
    below = np.where(surv <= 0.5)[0]
    return times[below[0]] if len(below) else np.inf
for p, name in [(0, "monthly"), (1, "annual")]:
    g = df[df.plan == p]
    tt, ss = kaplan_meier(g.months.to_numpy(), g.churned.to_numpy())
    s12 = ss[tt <= 12][-1]
    print(f"{name:8s}: still subscribed after 12 months {s12:.3f}, median survival {median_survival(tt, ss):.1f} months")

monthly : still subscribed after 12 months 0.112, median survival 3.2 months
annual  : still subscribed after 12 months 0.297, median survival 6.9 months


## 5. Comparing groups: the log-rank test

In [3]:
from scipy import stats
def logrank(t, e, g):
    O_minus_E, V = 0.0, 0.0
    for u in np.unique(t[e == 1]):
        at = t >= u; n_all, n1 = at.sum(), (at & (g == 1)).sum()
        d_all, d1 = ((t == u) & (e == 1)).sum(), ((t == u) & (e == 1) & (g == 1)).sum()
        O_minus_E += d1 - d_all * n1 / n_all
        if n_all > 1: V += d_all * (n1 / n_all) * (1 - n1 / n_all) * (n_all - d_all) / (n_all - 1)
    chi2 = O_minus_E ** 2 / V
    return chi2, stats.chi2.sf(chi2, 1)
chi2, p = logrank(df.months.to_numpy(), df.churned.to_numpy(), df.plan.to_numpy())
print(f"log-rank test, annual vs monthly: chi-squared {chi2:.1f}, p-value {p:.2g}")

log-rank test, annual vs monthly: chi-squared 55.0, p-value 1.2e-13


## 6. The Cox proportional hazards model

In [4]:
from scipy.optimize import minimize
Xc = df[["plan", "support"]].to_numpy(float); T, E = df.months.to_numpy(), df.churned.to_numpy()
def neg_partial_loglik(beta):
    eta = Xc @ beta; total = 0.0
    for i in np.where(E == 1)[0]:
        total += eta[i] - np.log(np.exp(eta[T >= T[i]]).sum())   # the risk set at that moment
    return -total
fit = minimize(neg_partial_loglik, np.zeros(2), method="BFGS")
for name, b in zip(["annual plan", "support tickets"], fit.x):
    print(f"{name:15s} coefficient {b:+.3f}   hazard ratio {np.exp(b):.2f}")
print("true coefficients: -0.8 and +0.25")

annual plan     coefficient -0.698   hazard ratio 0.50
support tickets coefficient +0.258   hazard ratio 1.29
true coefficients: -0.8 and +0.25


## 7. Evaluating survival models

In [5]:
def concordance(T, E, risk):
    ok = tot = 0
    for i in np.where(E == 1)[0]:
        later = T > T[i]                                           # pairs we can order: i churned first
        tot += later.sum(); ok += (risk[i] > risk[later]).sum() + 0.5 * (risk[i] == risk[later]).sum()
    return ok / tot
print("concordance index, Cox model:", round(concordance(T, E, Xc @ fit.x), 3))
print("concordance index, support tickets only:", round(concordance(T, E, Xc[:, 1]), 3))
print("concordance index, random scores:", round(concordance(T, E, rng.random(len(T))), 3))

concordance index, Cox model: 0.639
concordance index, support tickets only: 0.59
concordance index, random scores: 0.517


## 9. In Python: survival from a subscription table

In [6]:
import numpy as np
import pandas as pd

subs = pd.DataFrame({
    "signup":   pd.to_datetime(["2025-01-05", "2025-02-10", "2025-02-20", "2025-03-01", "2025-04-15", "2025-05-01", "2025-06-10", "2025-07-01"]),
    "cancelled": pd.to_datetime(["2025-04-01", None, "2025-05-20", "2025-03-30", None, "2025-09-15", None, "2025-08-01"])})
today = pd.Timestamp("2025-12-31")
subs["event"] = subs.cancelled.notna().astype(int)
subs["months"] = ((subs.cancelled.fillna(today) - subs.signup).dt.days / 30.44).round(1)
rows, s, at_risk = [], 1.0, len(subs)
for t, g in subs.sort_values("months").groupby("months"):
    d = g.event.sum()
    if d:
        s *= 1 - d / at_risk
        rows.append((t, at_risk, d, round(s, 3)))
    at_risk -= len(g)
print(subs[["months", "event"]].to_string(index=False))
print(pd.DataFrame(rows, columns=["month", "at risk", "churned", "survival"]).to_string(index=False))

 months  event
    2.8      1
   10.6      0
    2.9      1
    1.0      1
    8.5      0
    4.5      1
    6.7      0
    1.0      1
 month  at risk  churned  survival
   1.0        8        2     0.750
   2.8        6        1     0.625
   2.9        5        1     0.500
   4.5        4        1     0.375


## 12. Going deeper: checking proportional hazards, and restricted mean survival

In [7]:
import numpy as np
rng = np.random.default_rng(5); n = 4000
g = rng.integers(0, 2, n)
# Group 1's benefit fades: hazard ratio 0.4 in the first 6 months, 1.0 afterwards.
t_early = rng.exponential(1 / (0.15 * np.where(g == 1, 0.4, 1.0)))
t = np.where(t_early < 6, t_early, 6 + rng.exponential(1 / 0.15, n))
e = (t < 24).astype(int); t = np.minimum(t, 24)
def hazard(grp, lo, hi):                       # events per month at risk, between months lo and hi
    m = g == grp
    events = ((e[m] == 1) & (t[m] >= lo) & (t[m] < hi)).sum()
    exposure = np.clip(np.minimum(t[m], hi) - lo, 0, None).sum()
    return events / exposure
for lo, hi in [(0, 6), (6, 24)]:
    print(f"months {lo}–{hi}: hazard ratio, group 1 vs group 0, {hazard(1, lo, hi) / hazard(0, lo, hi):.2f}")

months 0–6: hazard ratio, group 1 vs group 0, 0.39
months 6–24: hazard ratio, group 1 vs group 0, 1.04


In [8]:
def km_curve(tt, ee):
    times, s, at, out = np.sort(np.unique(tt[ee == 1])), 1.0, None, []
    for u in times:
        s *= 1 - ((tt == u) & (ee == 1)).sum() / (tt >= u).sum(); out.append((u, s))
    return out
def rmst(curve, horizon):
    area, prev_t, prev_s = 0.0, 0.0, 1.0
    for u, s in curve:
        if u > horizon: break
        area += prev_s * (u - prev_t); prev_t, prev_s = u, s
    return area + prev_s * (horizon - prev_t)
for grp in [0, 1]:
    print(f"group {grp}: restricted mean survival over 12 months {rmst(km_curve(t[g == grp], e[g == grp]), 12):.2f} months")

group 0: restricted mean survival over 12 months 5.45 months
group 1: restricted mean survival over 12 months 7.71 months


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [9]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
# Simulated days on the market: fast sellers go quickly; one listing is still unsold (censored at 90 days).
df["days"] = np.where(df.sold_fast == 1, 10 + 4 * df.id, 40 + 5 * df.id).clip(max=90)
df["sold"] = (df.days < 90).astype(int)
def km(t, e):
    s, out = 1.0, []
    for u in np.unique(t[e == 1]):
        s *= 1 - ((t == u) & (e == 1)).sum() / (t >= u).sum(); out.append((int(u), round(float(s), 3)))
    return out
print("days:", df.days.tolist(), " sold:", df.sold.tolist())
print("Kaplan–Meier (day, share unsold):", km(df.days.to_numpy(), df.sold.to_numpy()))

days: [14, 18, 55, 26, 30, 70, 75, 80, 85, 90, 54, 58]  sold: [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1]
Kaplan–Meier (day, share unsold): [(14, 0.917), (18, 0.833), (26, 0.75), (30, 0.667), (54, 0.583), (55, 0.5), (58, 0.417), (70, 0.333), (75, 0.25), (80, 0.167), (85, 0.083)]


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [10]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: A Kaplan–Meier step

Multiply the survival by the share that survives this event time.

In [11]:
s, at_risk, events = 0.8, 10, 2
s_new = s - events  # ← multiply by the surviving share instead
check(s_new, 0.6400000000000001, hint='Multiply by 1 − d/n.')

❌ Not yet. Hint: Multiply by 1 − d/n.


### Challenge 2: A hazard ratio

Convert a Cox coefficient to a hazard ratio, rounded to 2 decimals.

In [12]:
import math
beta = 0.7
hr = beta  # ← exponentiate
check(hr, 2.01, hint='The hazard ratio is e to the power of beta.')

❌ Not yet. Hint: The hazard ratio is e to the power of beta.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. A Kaplan–Meier step** — the missing line:

```python
s_new = s * (1 - events / at_risk)
```

**2. A hazard ratio** — the missing line:

```python
hr = round(math.exp(beta), 2)
```